In [1]:
import os 
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

True

In [3]:
load_dotenv(override=True)
GROQ_API_KEY = (os.getenv("GROQ_API_KEY") or "").strip().strip('"')
HUGGINGFACE_API_KEY = (os.getenv("HF_TOKEN") or "").strip().strip('"')
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment variables")
if not HUGGINGFACE_API_KEY:
    raise ValueError("HUGGINGFACE_API_KEY not found in environment variables")

print("GROQ_API_KEY loaded successfully")
print("HUGGINGFACE_API_KEY loaded successfully")

GROQ_API_KEY loaded successfully
HUGGINGFACE_API_KEY loaded successfully


# Mem0 Memory

In [4]:
mem0_config = {
    "llm": {
        "provider": "groq",
        "config": {
            "model": "llama-3.3-70b-versatile",
            "api_key": GROQ_API_KEY,
            "temperature": 0.7,
            "max_tokens": 1000,
        },
    },
    "embedder": {
        "provider": "huggingface",
        "config": {
            "model": "all-MiniLM-L6-v2",
            "model_kwargs": {
                "token": False,
            },
        },
    },
    "vector_store": {
        "provider": "qdrant",
        "config": {
            "path": "./data/mem0-qdrant-notebook",
            "embedding_model_dims": 384,
        },
    },
}

In [5]:
from mem0 import Memory

m = Memory.from_config(mem0_config)

c:\Users\rambo\projects\cutomer_support_agent_langmem-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2676.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\rambo\projects\cutomer_support_agent_langmem-main\.venv\Lib\site-packages\mem0\embeddings\huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimen

In [ ]:
messages = [
    {"role": "user", "content": "Hi, I'm Alex. I love basketball and gaming."},
    {"role": "assistant", "content": "Hey Alex! I'll remember your interests."}
]
m.add(messages, user_id="alex")

print ("Added messages to memory.")

{'results': []}

In [14]:
results = m.search("What do you know about me?", filters={"user_id": "alex"})
print(results)

{'results': [{'id': 'e765babd-f670-44a6-bb3b-962faec661e0', 'memory': "User's name is Alex", 'hash': 'd4ce67b5c6d30432b784fdb03f91b5f6', 'metadata': None, 'score': 0.19328388945702046, 'created_at': '2026-04-21T01:53:38.777713+00:00', 'updated_at': '2026-04-21T01:53:38.777713+00:00', 'user_id': 'alex'}, {'id': 'a1e4bb8d-9da8-474e-96a8-16ca84b8a847', 'memory': 'User loves basketball', 'hash': '8a2750b98208518f1fd4fc50aac28a87', 'metadata': None, 'score': 0.12050989695412531, 'created_at': '2026-04-21T01:53:38.777713+00:00', 'updated_at': '2026-04-21T01:53:38.777713+00:00', 'user_id': 'alex'}]}


In [15]:
messages = [
    {"role": "user", "content": "Hi, I'm Yash. I love chess and programming."},
    {"role": "assistant", "content": "Hey Yash! I'll remember your interests."}
]
m.add(messages, user_id="yash")

{'results': [{'id': '4ae5ecce-0dc2-42b6-aa9a-c3684385a6cb',
   'memory': "User's name is Yash and they enjoy chess and programming",
   'event': 'ADD'}]}

In [16]:
m.search("What are yash's interests?", filters={"user_id": "yash"}, top_k=1)

{'results': [{'id': '4ae5ecce-0dc2-42b6-aa9a-c3684385a6cb',
   'memory': "User's name is Yash and they enjoy chess and programming",
   'hash': 'd760823193546b4e072655f674a8ac12',
   'metadata': None,
   'score': 0.48696082515748984,
   'created_at': '2026-04-21T02:06:02.990286+00:00',
   'updated_at': '2026-04-21T02:06:02.990286+00:00',
   'user_id': 'yash'}]}

# langchain GROQ


In [19]:
from langchain_groq import ChatGroq

In [22]:
from langchain_groq import ChatGroq
llm_groq = ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY, temperature=0.7, max_tokens=1000)


llm_groq.invoke("Hi! i am yash")

AIMessage(content='Nice to meet you, Yash. Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 41, 'total_tokens': 65, 'completion_time': 0.033798385, 'completion_tokens_details': None, 'prompt_time': 0.002215113, 'prompt_tokens_details': None, 'queue_time': 0.044462901, 'total_time': 0.036013498}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dadd3-2fd4-7180-8de8-b97ab8468eac-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 24, 'total_tokens': 65})

In [23]:
from langchain.agents import create_agent
from langchain.tools import tool